# Lezione 49 — Preference tuning di una politica

> **Modulo:** Feedback e preference training · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 48 (DPO), Lezione 40 (training a mano).
>
> Obiettivo pratico unico: usare DPO per **allineare una politica** di scoring
> delle memorie alle preferenze — la politica impara a preferire le stesse
> memorie che l'utente preferisce.

## Teoria essenziale

Fin qui: schema del feedback (45), coppie (46), reward (47), loss DPO (48). Ora
li mettiamo insieme. Una **politica** $\pi_\theta$ assegna a ogni memoria un
punteggio dalle sue feature; la trasformiamo in probabilita' con la softmax. La
**reference** $\pi_{ref}$ e' la politica di partenza. Applichiamo DPO: per ogni
coppia (chosen, rejected), alziamo la log-prob relativa della chosen.

A differenza della Lezione 48 (dove i log-prob erano parametri diretti), qui i
log-prob vengono da un **modello** $\pi_\theta(x) \propto \exp(\theta\cdot
\text{feat}(x))$: aggiornando $\theta$ cambiamo il comportamento su *tutte* le
memorie, non solo su quelle viste.

In [ ]:
import numpy as np

rng = np.random.default_rng(49)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# feature sintetiche per un pool di memorie; la preferenza vera dipende da theta*
d = 5
theta_star = rng.normal(size=d)
n_coppie = 300
Fw = rng.normal(size=(n_coppie, d))          # feature del chosen
Fl = rng.normal(size=(n_coppie, d))          # feature del rejected
# le etichette (chi e' chosen) seguono theta* con rumore (Bradley-Terry)
pref_corretta = rng.random(n_coppie) < sigmoid((Fw - Fl) @ theta_star)
# dove pref_corretta e' False, il "vero" chosen era l'altro: scambio
Fw2 = np.where(pref_corretta[:, None], Fw, Fl)
Fl2 = np.where(pref_corretta[:, None], Fl, Fw)
Fw, Fl = Fw2, Fl2
print("coppie di preferenza:", n_coppie, "| dim feature:", d)

In [ ]:
# politica log pi_theta(x) = theta·feat(x) (a meno di una costante di softmax
# che si CANCELLA nella differenza chosen-rejected di DPO)
theta = rng.normal(size=d) * 0.01   # politica addestrabile (quasi nulla)
theta_ref = theta.copy()            # reference = politica iniziale
beta = 1.0
lr = 0.5

def margine_dpo(theta):
    # (log pi_theta(w) - log pi_ref(w)) - (log pi_theta(l) - log pi_ref(l))
    return ((Fw @ theta - Fw @ theta_ref) - (Fl @ theta - Fl @ theta_ref))

def accuratezza(theta):
    return float((Fw @ theta > Fl @ theta).mean())

print(f"accuratezza iniziale della politica: {accuratezza(theta):.2f}  (~0.5 = caso)")
for passo in range(300):
    m = margine_dpo(theta)
    p = sigmoid(beta * m)
    grad = -beta * ((1 - p)[:, None] * (Fw - Fl)).mean(axis=0)
    theta -= lr * grad
    if passo % 100 == 0 or passo == 299:
        L = -np.log(sigmoid(beta * margine_dpo(theta)) + 1e-12).mean()
        print(f"passo {passo:3d}: loss {L:.4f}  accuratezza {accuratezza(theta):.2f}")

Leggi la salita: la politica, partita dalla reference (accuratezza ~0.5),
impara a preferire le memorie che l'utente preferisce. E poiche' e' un *modello*
sulle feature, generalizza a memorie nuove.

In [ ]:
# generalizzazione: su coppie MAI viste, la politica preferisce ancora bene?
Fw_test = rng.normal(size=(200, d))
Fl_test = rng.normal(size=(200, d))
pref_test = rng.random(200) < sigmoid((Fw_test - Fl_test) @ theta_star)
Fw_t = np.where(pref_test[:, None], Fw_test, Fl_test)
Fl_t = np.where(pref_test[:, None], Fl_test, Fw_test)
acc_test = float((Fw_t @ theta > Fl_t @ theta).mean())
cos = float(theta @ theta_star / (np.linalg.norm(theta) * np.linalg.norm(theta_star)))
print(f"accuratezza su coppie NUOVE: {acc_test:.2f}")
print(f"allineamento della politica con la preferenza vera (coseno): {cos:.3f}")

## Il progetto: Memory AI Lab, passo 49

In [ ]:
def preference_tuning(Fw, Fl, beta=1.0, lr=0.5, passi=300):
    d = Fw.shape[1]
    theta = np.zeros(d)
    ref = theta.copy()
    for _ in range(passi):
        m = (Fw @ theta - Fw @ ref) - (Fl @ theta - Fl @ ref)
        p = sigmoid(beta * m)
        theta -= lr * (-beta * ((1 - p)[:, None] * (Fw - Fl)).mean(axis=0))
    return theta

th = preference_tuning(Fw, Fl)
acc = float((Fw @ th > Fl @ th).mean())
# controllo di non-regressione: la politica tunata batte il caso e generalizza
assert acc > 0.7, acc
assert float((Fw_t @ th > Fl_t @ th).mean()) > 0.65
print(f"politica tunata: accuratezza train {acc:.2f}")

## Riepilogo (max 8 punti)

1. Preference tuning mette insieme feedback → coppie → DPO su una **politica**.
2. La politica $\pi_\theta(x)\propto\exp(\theta\cdot\text{feat}(x))$ da' un
   punteggio dalle feature.
3. La **reference** e' la politica di partenza.
4. DPO alza la log-prob relativa della chosen su ogni coppia.
5. La costante di softmax si **cancella** nella differenza chosen-rejected.
6. Essendo un modello, **generalizza** a memorie nuove.
7. L'accuratezza sale sia sul train sia su coppie mai viste.
8. La politica si allinea alla preferenza vera senza averla mai vista.

## Quiz

1. Perche' la politica generalizza a memorie non viste, a differenza della
   Lezione 48?
2. Cosa fa la reference in questo tuning?
3. Perche' la costante di normalizzazione della softmax non conta in DPO?

*(Risposte: 1. perche' e' un modello sulle feature, non un log-prob per elemento;
2. ancora la politica al punto di partenza, evitando derive eccessive; 3. si
elide nella differenza (log pi_w - log pi_l), essendo identica per entrambi.)*